In [1]:
import pandas as pd
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/CADE

/content/drive/My Drive/CADE


## Classification

In [4]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_classification.csv')

# Known labels across the dataset
LABELS = ['walking', 'jogging', 'sitting', 'standing', 'upstairs', 'downstairs',
          'no freeze', 'freeze', 'normal', 'abnormal', 'positive', 'negative',
          'supraventricular', 'premature ventricular', 'fusion', 'unclassifiable',
          'atrial premature', 'premature ventricular contraction', 'right bundle branch block',
          'left bundle branch block', 'paced']
# extract the last label of the sentence
def extract_label(text):
    text = str(text).lower() # converts everything to lowercase
    # Find the last occurrence of a known label
    best = None
    best_pos = -1
    for label in sorted(LABELS, key=len, reverse=True):  # longest first, prevents "ventricular" from being caught when the full label is actually "premature ventricular."
        pos = text.rfind(label) # rfind(): returns the index where is string is found. If not found, it returns -1
        if pos != -1 and pos > best_pos: # If a label is found later in the text than any previously found label, the script "overwrites" the old one
            best_pos = pos
            best = label
    return best

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL) # regular expression
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_label(true_text)
    model_label = extract_label(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        if len(mismatches) < 5:
            mismatches.append((i, true_label, model_label, str(row['model_response'])[:100]))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nSample mismatches:")
for idx, t, m, resp in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

# Check for any None labels
none_true = sum(1 for _, row in df.iterrows() if extract_label(str(row['QA_list'])) is None)
none_model = sum(1 for _, row in df.iterrows() if extract_label(str(row['model_response'])) is None)
print(f"\nUnparsed true labels: {none_true}, Unparsed model labels: {none_model}")
if none_model > 0:
    for i, row in df.iterrows():
        if extract_label(str(row['model_response'])) is None:
            print(f"  Row {i} model_response: {str(row['model_response'])[:200]}")

Correct: 294/400
Accuracy: 73.50%

Sample mismatches:
  Row 0: true='sitting' vs model='walking'
  Row 8: true='jogging' vs model='downstairs'
  Row 9: true='jogging' vs model='walking'
  Row 13: true='sitting' vs model='standing'
  Row 15: true='jogging' vs model='walking'

Unparsed true labels: 0, Unparsed model labels: 0


## Anomaly Detection

In [5]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_anomaly.csv')
print(f"Total rows: {len(df)}")

def extract_label(text):
    """Extract anomaly detection label: 'normal' or 'anomaly'."""
    text = str(text).lower()
    if 'no anomaly' in text or 'normal point' in text or 'no anomalies' in text:
        return 'normal'
    if 'anomaly point' in text or 'anomaly' in text or 'anomalies' in text:
        return 'anomaly'
    return None

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_label(true_text)
    model_label = extract_label(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

Total rows: 400
Correct: 248/400
Accuracy: 62.00%

Mismatches (152):
  Row 0: true='normal' vs model='anomaly'
  Row 2: true='normal' vs model='anomaly'
  Row 3: true='normal' vs model='anomaly'
  Row 8: true='normal' vs model='anomaly'
  Row 9: true='normal' vs model='anomaly'
  Row 12: true='normal' vs model='anomaly'
  Row 15: true='normal' vs model='anomaly'
  Row 16: true='normal' vs model='anomaly'
  Row 20: true='normal' vs model='anomaly'
  Row 23: true='normal' vs model='anomaly'
  Row 25: true='normal' vs model='anomaly'
  Row 29: true='normal' vs model='anomaly'
  Row 32: true='normal' vs model='anomaly'
  Row 33: true='normal' vs model='anomaly'
  Row 38: true='normal' vs model='anomaly'
  Row 39: true='normal' vs model='anomaly'
  Row 40: true='normal' vs model='anomaly'
  Row 41: true='normal' vs model='anomaly'
  Row 42: true='normal' vs model='anomaly'
  Row 43: true='normal' vs model='anomaly'
  Row 44: true='normal' vs model='anomaly'
  Row 49: true='normal' vs model=

## Multiple Choice

In [6]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_multiple_choice.csv')
print(f"Total rows: {len(df)}")

def extract_choice(text):
    """Extract multiple choice answer (A/B/C/D or True/False) from text."""
    text = str(text).strip()
    # Match leading letter choice like "A)", "a)", "B.", "C)" etc.
    m = re.match(r'^([A-Da-d])[).\s]', text) #
    if m:
        return m.group(1).upper() # catch the first parentheses of m (e.g., "b") and converts any lowercase letter to uppercase
    # "Answer: (b)" or "answer is (A)" or "option (c)"
    m = re.search(r'(?:answer|option)[:\s]+\(?([A-Da-d])\)?', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # "The answer is (A)"
    m = re.search(r'the answer is\s*\(?([A-Da-d])\)?', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    # True/False style answers in multiple choice
    if re.match(r'^true', text, re.IGNORECASE):
        return 'TRUE'
    if re.match(r'^false', text, re.IGNORECASE):
        return 'FALSE'
    # Last resort: find first standalone letter with parentheses. Like (A, B, C, or D) or (a, b, c, d)
    m = re.search(r'\(([A-Da-d])\)', text)
    if m:
        return m.group(1).upper()
    return None

correct = 0
total = len(df)
mismatches = []
unparsed = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_choice(true_text)
    model_label = extract_choice(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))
    if model_label is None:
        unparsed.append((i, str(row['model_response'])[:150]))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")
if unparsed:
    print(f"\nUnparsed model responses ({len(unparsed)}):")
    for idx, r in unparsed:
        print(f"  Row {idx}: {r}")

Total rows: 397
Correct: 172/397
Accuracy: 43.32%

Mismatches (225):
  Row 1: true='B' vs model='C'
  Row 3: true='A' vs model='B'
  Row 4: true='C' vs model='B'
  Row 5: true='C' vs model='B'
  Row 6: true='C' vs model='B'
  Row 7: true='D' vs model='C'
  Row 9: true='A' vs model='C'
  Row 10: true='A' vs model='B'
  Row 14: true='B' vs model='TRUE'
  Row 16: true='None' vs model='None'
  Row 17: true='B' vs model='C'
  Row 18: true='A' vs model='B'
  Row 19: true='D' vs model='C'
  Row 20: true='A' vs model='B'
  Row 21: true='A' vs model='B'
  Row 23: true='B' vs model='D'
  Row 24: true='B' vs model='C'
  Row 25: true='A' vs model='B'
  Row 27: true='D' vs model='C'
  Row 29: true='None' vs model='None'
  Row 30: true='A' vs model='B'
  Row 32: true='A' vs model='B'
  Row 33: true='A' vs model='B'
  Row 35: true='A' vs model='B'
  Row 37: true='A' vs model='C'
  Row 38: true='A' vs model='C'
  Row 40: true='B' vs model='A'
  Row 42: true='B' vs model='C'
  Row 44: true='A' vs model

## True/False Question

In [7]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_true_false.csv')
print(f"Total rows: {len(df)}")

def extract_tf(text):
    """Extract True/False label from text."""
    text = str(text).strip().lower()
    # Check the leading word first
    if re.match(r'^(true|yes)', text):
        return 'true'
    if re.match(r'^(false|no[^a-z])', text):
        return 'false'
    # Fallback: search anywhere (only if unambiguous)
    if 'true' in text and 'false' not in text:
        return 'true'
    if 'false' in text and 'true' not in text:
        return 'false'
    return None

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_tf(true_text)
    model_label = extract_tf(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

Total rows: 400
Correct: 280/400
Accuracy: 70.00%

Mismatches (120):
  Row 4: true='true' vs model='false'
  Row 8: true='true' vs model='false'
  Row 13: true='false' vs model='true'
  Row 14: true='true' vs model='false'
  Row 15: true='true' vs model='false'
  Row 22: true='true' vs model='false'
  Row 40: true='true' vs model='false'
  Row 42: true='false' vs model='true'
  Row 44: true='true' vs model='false'
  Row 46: true='true' vs model='false'
  Row 47: true='false' vs model='true'
  Row 49: true='false' vs model='true'
  Row 54: true='true' vs model='false'
  Row 56: true='true' vs model='false'
  Row 57: true='true' vs model='false'
  Row 62: true='false' vs model='true'
  Row 67: true='true' vs model='false'
  Row 71: true='true' vs model='false'
  Row 79: true='true' vs model='false'
  Row 82: true='false' vs model='true'
  Row 88: true='true' vs model='false'
  Row 92: true='true' vs model='false'
  Row 96: true='true' vs model='false'
  Row 98: true='true' vs model='fals

## Forecasting

In [8]:
"""
Evaluation Script: MSE for Forecasting Task
=============================================
Computes Mean Squared Error between the true answer and model response
for time series forecasting results.

Input:  results_forecasting.csv
        - Column: QA_list     (contains the true answer list/value after "answer" key)
        - Column: model_response (contains the model's predicted list/value)

Output: MSE statistics printed to console.
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    """Extract a list of floats from text containing a bracketed list like [1.0, 2.0, ...]."""
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"") # remove whitespace and '"
        try:
            result.append(float(item)) # transform into float
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None] # remove the None value from list


def extract_single_number(text):
    """
    Fallback: extract a single numeric value from free-text responses.
    Used when the answer/response is a single predicted value rather than a list.
    """
    nums = re.findall(r'[-+]?\d*\.?\d+', text) # find numeric value
    if nums:
        return [float(nums[-1])] # extract and return the last numeric value
    return None


def parse_answer(qa_text):
    """Extract the answer from the QA_list column."""
    if '"answer"' not in qa_text:
        return None

    answer_part = qa_text.split('"answer"')[-1] # take the last part {"question": "...", "answer": "[1, 2, 3]"})

    # Try list extraction first
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list

    # Fallback: single number from the answer text
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL) # extract content after answer within ""
    if ans_match:
        return extract_single_number(ans_match.group(1))

    return None


def parse_model_response(model_text):
    """Extract the predicted values from the model_response column."""
    # Try list extraction first
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list

    # Fallback: single number
    return extract_single_number(model_text)


def compute_mse_forecasting(csv_path):
    """
    Parse the forecasting CSV and compute MSE between answer and model_response.

    For each row:
      1. Extract the answer (list or single value) from the QA_list column.
      2. Extract the prediction (list or single value) from the model_response column.
      3. Align by index using min(len(answer), len(model)) to handle length mismatches.
      4. Compute squared errors for each aligned pair.

    Returns a dict with detailed statistics.
    """
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)

        all_squared_errors = []
        row_mses = []
        row_details = []
        skipped_rows = []
        row_count = 0

        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column

            # --- Extract answer and model response ---
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)

            # --- Validate ---
            if answer_list is None or model_list is None:
                skipped_rows.append((row_count, "Could not parse list(s)"))
                row_count += 1
                continue

            if len(answer_list) == 0 or len(model_list) == 0:
                skipped_rows.append((row_count, "Empty list after parsing"))
                row_count += 1
                continue

            # --- Compute squared errors (aligned by position) ---
            min_len = min(len(answer_list), len(model_list))
            se = [(answer_list[i] - model_list[i]) ** 2 for i in range(min_len)]
            row_mse = np.mean(se)

            row_mses.append(row_mse)
            all_squared_errors.extend(se)
            row_details.append((row_count, len(answer_list), len(model_list), row_mse))

            row_count += 1

    # --- Aggregate statistics ---
    results = {
        "total_rows": row_count,
        "rows_used": len(row_mses),
        "rows_skipped": len(skipped_rows),
        "total_value_pairs": len(all_squared_errors),
        "mse_all_pairs": float(np.mean(all_squared_errors)) if all_squared_errors else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "row_details": row_details,
        "skipped_rows": skipped_rows,
    }

    # Matched-length subset
    matched = [d for d in row_details if d[1] == d[2]]
    results["rows_matched_length"] = len(matched)
    results["mse_matched_length"] = float(np.mean([d[3] for d in matched])) if matched else None

    return results


def print_report(results):
    """Pretty-print the evaluation report."""
    print("=" * 60)
    print("  FORECASTING MSE EVALUATION REPORT")
    print("=" * 60)
    print(f"  Total rows in CSV:          {results['total_rows']}")
    print(f"  Rows successfully parsed:   {results['rows_used']}")
    print(f"  Rows skipped (unparseable): {results['rows_skipped']}")
    print(f"  Total value pairs compared: {results['total_value_pairs']}")
    print("-" * 60)
    print(f"  MSE (all value pairs):      {results['mse_all_pairs']:.6f}")
    print(f"  MSE (avg of per-row MSEs):  {results['mse_avg_per_row']:.6f}")
    print("-" * 60)
    print(f"  Rows with matching lengths: {results['rows_matched_length']}/{results['rows_used']}")
    if results['mse_matched_length'] is not None:
        print(f"  MSE (matched-length only):  {results['mse_matched_length']:.6f}")
    print("=" * 60)

    # Top 10 rows by MSE
    sorted_rows = sorted(results['row_details'], key=lambda x: x[3], reverse=True)
    print("\n  Top 10 rows by per-row MSE:")
    print(f"  {'Row':<6} {'Ans Len':<10} {'Model Len':<10} {'Row MSE':>15}")
    print("  " + "-" * 45)
    for r, alen, mlen, mse in sorted_rows[:10]:
        flag = " *" if alen != mlen else ""
        print(f"  {r:<6} {alen:<10} {mlen:<10} {mse:>15.4f}{flag}")
    print("  (* = length mismatch)")

    # Skipped rows
    if results['skipped_rows']:
        print(f"\n  Skipped rows:")
        for idx, reason in results['skipped_rows']:
            print(f"    Row {idx}: {reason}")


if __name__ == "__main__":
    CSV_PATH = "/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_forecasting.csv"
    results = compute_mse_forecasting(CSV_PATH)
    print_report(results)

  FORECASTING MSE EVALUATION REPORT
  Total rows in CSV:          379
  Rows successfully parsed:   378
  Rows skipped (unparseable): 1
  Total value pairs compared: 7026
------------------------------------------------------------
  MSE (all value pairs):      1012960.910499
  MSE (avg of per-row MSEs):  1104244.589270
------------------------------------------------------------
  Rows with matching lengths: 176/378
  MSE (matched-length only):  2065466.032074

  Top 10 rows by per-row MSE:
  Row    Ans Len    Model Len          Row MSE
  ---------------------------------------------
  369    12         12          271966485.3635
  231    30         30           69742994.6000
  43     28         26           29689274.4231 *
  291    32         28           12667647.9643 *
  377    12         12           12207732.0000
  178    28         26            3689873.6923 *
  94     26         26            3042083.8077
  18     25         26            2038531.0800 *
  311    11         11  

## Imputation

In [9]:
"""
Evaluation Script: MSE for Imputation Task
============================================
Computes Mean Squared Error between the true answer and model response
for time series imputation results.

Input:  results_imputation.csv
        - Column: QA_list     (contains the true answer list after "answer" key)
        - Column: model_response (contains the model's predicted list)

Output: MSE statistics printed to console.
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    """Extract a list of floats from text containing a bracketed list like [1.0, 2.0, ...]."""
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)  # e.g. 'X' or non-numeric tokens
    return [x for x in result if x is not None]


def compute_mse_imputation(csv_path):
    """
    Parse the imputation CSV and compute MSE between answer and model_response.

    For each row:
      1. Extract the answer list from the QA_list column (after the "answer" key).
      2. Extract the predicted list from the model_response column.
      3. Align by index using min(len(answer), len(model)) to handle length mismatches.
      4. Compute squared errors for each aligned pair.

    Returns a dict with detailed statistics.
    """
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)

        all_squared_errors = []   # every individual (answer_i - model_i)^2
        row_mses = []             # per-row MSE values
        row_details = []          # (row_index, answer_len, model_len, row_mse)
        skipped_rows = []         # rows that couldn't be parsed
        row_count = 0

        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column

            # --- Extract answer list ---
            answer_list = None
            if '"answer"' in qa_text:
                answer_part = qa_text.split('"answer"')[-1]
                answer_list = extract_list_from_text(answer_part)

            # --- Extract model response list ---
            model_list = extract_list_from_text(model_text)

            # --- Validate ---
            if answer_list is None or model_list is None:
                skipped_rows.append((row_count, "Could not parse list(s)"))
                row_count += 1
                continue

            if len(answer_list) == 0 or len(model_list) == 0:
                skipped_rows.append((row_count, "Empty list after parsing"))
                row_count += 1
                continue

            # --- Compute squared errors (aligned by position) ---
            min_len = min(len(answer_list), len(model_list))
            se = [(answer_list[i] - model_list[i]) ** 2 for i in range(min_len)]
            row_mse = np.mean(se)

            row_mses.append(row_mse)
            all_squared_errors.extend(se)
            row_details.append((row_count, len(answer_list), len(model_list), row_mse))

            row_count += 1

    # --- Aggregate statistics ---
    results = {
        "total_rows": row_count,
        "rows_used": len(row_mses),
        "rows_skipped": len(skipped_rows),
        "total_value_pairs": len(all_squared_errors),
        "mse_all_pairs": float(np.mean(all_squared_errors)) if all_squared_errors else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "row_details": row_details,
        "skipped_rows": skipped_rows,
    }

    # Matched-length subset
    matched = [d for d in row_details if d[1] == d[2]]
    results["rows_matched_length"] = len(matched)
    results["mse_matched_length"] = float(np.mean([d[3] for d in matched])) if matched else None

    return results


def print_report(results):
    """Pretty-print the evaluation report."""
    print("=" * 60)
    print("  IMPUTATION MSE EVALUATION REPORT")
    print("=" * 60)
    print(f"  Total rows in CSV:          {results['total_rows']}")
    print(f"  Rows successfully parsed:   {results['rows_used']}")
    print(f"  Rows skipped (unparseable): {results['rows_skipped']}")
    print(f"  Total value pairs compared: {results['total_value_pairs']}")
    print("-" * 60)
    print(f"  MSE (all value pairs):      {results['mse_all_pairs']:.6f}")
    print(f"  MSE (avg of per-row MSEs):  {results['mse_avg_per_row']:.6f}")
    print("-" * 60)
    print(f"  Rows with matching lengths: {results['rows_matched_length']}/{results['rows_used']}")
    if results['mse_matched_length'] is not None:
        print(f"  MSE (matched-length only):  {results['mse_matched_length']:.6f}")
    print("=" * 60)

    # Top 10 rows by MSE
    sorted_rows = sorted(results['row_details'], key=lambda x: x[3], reverse=True)
    print("\n  Top 10 rows by per-row MSE:")
    print(f"  {'Row':<6} {'Ans Len':<10} {'Model Len':<10} {'Row MSE':>15}")
    print("  " + "-" * 45)
    for r, alen, mlen, mse in sorted_rows[:10]:
        flag = " *" if alen != mlen else ""
        print(f"  {r:<6} {alen:<10} {mlen:<10} {mse:>15.4f}{flag}")
    print("  (* = length mismatch)")

    # Skipped rows
    if results['skipped_rows']:
        print(f"\n  Skipped rows:")
        for idx, reason in results['skipped_rows']:
            print(f"    Row {idx}: {reason}")


if __name__ == "__main__":
    CSV_PATH = "/content/drive/My Drive/CADE/results/qwen+full_finetuning/results_imputation.csv"
    results = compute_mse_imputation(CSV_PATH)
    print_report(results)

  IMPUTATION MSE EVALUATION REPORT
  Total rows in CSV:          400
  Rows successfully parsed:   394
  Rows skipped (unparseable): 6
  Total value pairs compared: 64593
------------------------------------------------------------
  MSE (all value pairs):      2663201.773972
  MSE (avg of per-row MSEs):  2391803.003720
------------------------------------------------------------
  Rows with matching lengths: 228/394
  MSE (matched-length only):  3303.683718

  Top 10 rows by per-row MSE:
  Row    Ans Len    Model Len          Row MSE
  ---------------------------------------------
  64     195        187         894583254.0374 *
  12     142        84           17004921.0238 *
  8      113        112           7442345.9554 *
  347    122        34            5931676.0000 *
  73     200        184           4253694.6196 *
  56     105        103           2319845.3495 *
  278    144        143           2093957.6503 *
  142    179        27            1829093.1852 *
  5      256       